In [1]:
import pandas as pd

In [ ]:
vi_filename = "data/all_vis_all_cams_with_yield_and_weather.xlsx"
temperature_filename = "data/lepton_temperature.xlsx"
merged_vi_temp_filename = "data/all_vi_temp_weather_with_yield.xlsx"

In [ ]:
# Merging vis and lepton temperature values

vi_df = pd.read_excel(vi_filename)
temperature_df = pd.read_excel(temperature_filename)

combined_vi_and_temp_df = pd.DataFrame()

vi_df['date'] = pd.to_datetime(vi_df['date'], format='%Y-%m-%d')
temperature_df['date'] = pd.to_datetime(temperature_df['date'], format='%Y-%m-%d')
temperature_df = temperature_df.rename(columns={
    "plot": "plot_location"
})

temperature_df['plot_location'] = temperature_df['plot_location'].str.replace('_', '-', regex=False)

merged_df = vi_df.merge(
    temperature_df,
    on=["cam_no", "plot_location", "date"],
    how="left",
)

print(merged_df.head())
print(merged_df.shape)

merged_df.to_excel(merged_vi_temp_filename, index=False)


In [ ]:
df = pd.read_excel(merged_vi_temp_filename)
print(df.shape)


In [ ]:

vi_features = ['ndvi_noir_rgb_method', 'gndvi_noir_rgb_method', 'rdvi_noir_rgb_method', 'savi_noir_rgb_method', 'sr_noir_rgb_method','evi_noir_rgb_method', 'cigreen_noir_rgb_method' , 'gli', 'vari']
weather_features = ['Solar', 'Precipitation', 'AirTemp', 'Vapor_Pressure', 'AtmPressure', 'RelHumidity']
canopy_temp_features = ['lepton_mean_temp', 'lepton_min_temp', 'lepton_max_temp']
features = vi_features + canopy_temp_features + weather_features
output_variable = 'GYLD_kg_m2'

In [ ]:

# df_hour13 = df[(df['hour'] > 10) & (df['hour'] < 14)].copy() # 11.** 12.**, 13.**
# df_hour10 = df[(df['hour'] > 8) & (df['hour'] < 11)].copy() # 9.**, 10.**

df_hour13 = df[(df['hour_group'] == 13 )].copy()
df_hour10 = df[(df['hour_group'] == 10 )].copy()

# Filter yield greater than 0
df_hour13 = df_hour13[(df_hour13[output_variable] > 0)]
df_hour10 = df_hour10[(df_hour10[output_variable] > 0)]

In [ ]:
# Checking duplicates
duplicates = df_hour13.groupby(['plot_id', 'date']).size().reset_index(name='count')
duplicates = duplicates[duplicates['count'] > 1]

print(duplicates) # 0 results, Good!

In [ ]:
df_hour10 = df_hour10[["plot_id", "date"] + features + [output_variable]].copy()

df_hour13 = df_hour13[["plot_id", "date"] + features + [output_variable]].copy()

In [ ]:
print(df_hour13.describe())
print(f"Remaining rows: {len(df_hour13)}")

blank_count = df_hour13['lepton_mean_temp'].isna().sum()
print(blank_count)


In [ ]:
df_hour13['date'] = pd.to_datetime(df_hour13['date'])
df_hour13 = df_hour13.sort_values([  'plot_id', 'date',])

df_hour10['date'] = pd.to_datetime(df_hour10['date'])
df_hour10 = df_hour10.sort_values([  'plot_id', 'date',])
print(df_hour13.head(48))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_available_data(dataset):
    df_avail = dataset.copy()

    for _feature in features:
        presence = df_avail.pivot_table(
            index='plot_id',
            columns='date',
            values=_feature,
            aggfunc='size',
            fill_value=0
        )
        presence = presence.map(lambda x: 1 if x > 0 else 0)

        plt.figure(figsize=(15, 8))
        sns.heatmap(presence, cmap="Greens", cbar=False)
        plt.title(f"Data Availability by Plot and Date ({_feature})")
        plt.xlabel("Date")
        plt.ylabel("Plot ID")
        plt.show()

plot_available_data(df_hour13)

In [ ]:
# Count how many rows per plot_id and date
duplicates = df_hour13.groupby(['plot_id', 'date']).size().reset_index(name='count')

# Filter only those with more than 1 NDVI value
duplicates = duplicates[duplicates['count'] > 1]

print(duplicates) # 0 results, Good!

In [ ]:
blank_count = df_hour13['lepton_mean_temp'].isna().sum()
print(blank_count)

print(df_hour13.shape)

Outliers

In [ ]:
# Checking outliers
for feature in features:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df_hour13[feature])
    plt.title(feature.upper() + " Boxplot - Outliers Visible")
    plt.show()

In [ ]:
# Removing outliers for each plot
def remove_outliers_iqr_all_features(group):
    for _feature in features:
        Q1 = group[_feature].quantile(0.25)
        Q3 = group[_feature].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1 * IQR
        upper_bound = Q3 + 1 * IQR
        group = group[(group[_feature] >= lower_bound) & (group[_feature] <= upper_bound)]
    return group

# Clean per plot_id
df_clean = df_hour13.groupby('plot_id', group_keys=False).apply(remove_outliers_iqr_all_features)

In [ ]:
# Plot the NDVI over the time
plot_to_check = "8_w"

def plot_VI_over_time_per_plot(plot_no, feature_name, vi_name, non_interpolated_dataset, with_interpolation=False, interpolation_method="", interpolated_dataset=None):

    plot_data = non_interpolated_dataset[non_interpolated_dataset['plot_id'] == plot_no]

    plt.figure(figsize=(12,6))

    if with_interpolation:
        if interpolated_dataset is None:
            raise ValueError("Interpolated dataset is missing in the arguments")
        plot_data_interpolated = interpolated_dataset[interpolated_dataset['plot_id'] == plot_no]
        plt.plot(plot_data_interpolated['date'], plot_data_interpolated[feature_name], label=interpolation_method + ' Interpolated ' + vi_name)
        plt.scatter(plot_data_interpolated['date'], plot_data_interpolated[feature_name], label=interpolation_method + ' Interpolated ' + vi_name)
    else:
        plt.plot(plot_data['date'], plot_data[feature_name], label=vi_name + ' Before Interpolation')

    plt.scatter(plot_data['date'], plot_data[feature_name], color='red', label=vi_name + ' Before Interpolation')
    plt.title(f"{vi_name} Time Series for Plot {plot_no}")
    plt.xlabel("Date")
    plt.ylabel("NDVI")
    plt.legend()
    plt.show()

plot_VI_over_time_per_plot(plot_to_check, 'ndvi_noir_rgb_method' , "NDVI", df_hour13)
plot_VI_over_time_per_plot(plot_to_check, 'gndvi_noir_rgb_method' , "GNDVI",df_hour13)
plot_VI_over_time_per_plot(plot_to_check, 'lepton_mean_temp' , "Canopy Temperature",df_hour13)
plot_VI_over_time_per_plot(plot_to_check, 'AirTemp' , "Air Temperature",df_hour13)

In [ ]:
df_hour13 = df_hour13[(df_hour13['date'] >= '2024-05-01') & (df_hour13['date'] <= '2024-06-30')]
plot_available_data(df_hour13)

In [ ]:
all_dates = pd.date_range(df_hour13['date'].min(), df_hour13['date'].max(), freq="D")

num_na_rows = df_hour13.isna().any(axis=1).sum()
print(f"df_hour13 Rows with at least one NaN: {num_na_rows}")

blank_count = df_hour13[['lepton_mean_temp', 'AirTemp']].isna().any(axis=1).sum()
print(f"df_linear_interpolated lepton_mean_temp or weather NaN: {blank_count}")


print(df_hour13.shape)


In [ ]:
from scipy.interpolate import PchipInterpolator

def interpolate_plot(plot, method,features_to_interpolate,  doSmoothing= False):
    plot = plot.set_index('date').sort_index()
    plot = plot.reindex(all_dates)
    plot['plot_id'] = plot['plot_id'].ffill().bfill()

    if method == "pchip":
        for _feature in features_to_interpolate:
            vi = plot[_feature]
            if vi.notna().sum() >= 2:
                valid = vi.dropna()
                interpolator = PchipInterpolator(valid.index, valid.values)
                plot[_feature] = interpolator(plot.index)
            else:
                # Not enough data → forward/backward fill
                plot[_feature] = vi.ffill().bfill()
    elif method == "polynomial" or method == "spline":
        if doSmoothing:
            plot[features_to_interpolate] = (plot[features_to_interpolate].interpolate(method=method, order=2, limit_direction='both')
                                        .rolling(window=3, min_periods=1, center=True).mean())
        else:
            plot[features_to_interpolate] = plot[features_to_interpolate].interpolate(method=method, order=2, limit_direction='both')
    else:
        if doSmoothing:
            plot[features_to_interpolate] = (plot[features_to_interpolate].interpolate(method=method,limit_direction='both')
                                        .rolling(window=3, min_periods=1, center=True).mean())
        else:
            plot[features_to_interpolate] = plot[features_to_interpolate].interpolate(method=method,limit_direction='both')

    plot[output_variable] = plot[output_variable].ffill().bfill()  # same yield for whole plot
    plot = plot.reset_index().rename(columns={'index': 'date'})
    return plot


In [ ]:

# Apply linear interpolation
# Without smoothing
df_linear_interpolated = (
    df_hour13
    .groupby('plot_id', group_keys=False)
    .apply(interpolate_plot, method='linear', features_to_interpolate=features)
    .reset_index(drop=True)
)
num_na_rows = df_linear_interpolated.isna().any(axis=1).sum()
print(f"df_linear_interpolated Rows with at least one NaN: {num_na_rows}")

blank_count = df_linear_interpolated['ndvi_noir_rgb_method'].isna().sum()
print(f"df_linear_interpolated ndvi_noir_rgb_method NaN: {blank_count}")

blank_count = df_linear_interpolated[['lepton_mean_temp', 'AirTemp']].isna().any(axis=1).sum()
print(f"df_linear_interpolated lepton_mean_temp or weather NaN: {blank_count}")

blank_count = df_linear_interpolated[['AirTemp']].isna().any(axis=1).sum()
print(f"df_linear_interpolated AirTemp  NaN: {blank_count}")


print(df_linear_interpolated.shape)

# With smoothing
df_linear_interpolated_and_smoothed = (
    df_hour13
    .groupby('plot_id', group_keys=False)
    .apply(interpolate_plot, method='linear', doSmoothing=True, features_to_interpolate=features)
    .reset_index(drop=True)
)
num_na_rows = df_linear_interpolated_and_smoothed.isna().any(axis=1).sum()
print(f"df_linear_interpolated_and_smoothed Rows with at least one NaN: {num_na_rows}")


In [ ]:
# Apply time interpolation
df_time_interpolated = (
    df_hour13
    .groupby('plot_id', group_keys=False)
    .apply(interpolate_plot, method='time', features_to_interpolate=features)
    .reset_index(drop=True)
)
num_na_rows = df_time_interpolated.isna().any(axis=1).sum()
print(f"df_time_interpolated Rows with at least one NaN: {num_na_rows}")
# Linear and time interpolations looks the same

In [ ]:
# Apply polynomial interpolation
df_polynomial_interpolated = (
    df_hour13
    .groupby('plot_id', group_keys=False)
    .apply(interpolate_plot, method='polynomial', features_to_interpolate=features)
    .reset_index(drop=True)
)
plot_to_check = "10_d"
num_na_rows = df_polynomial_interpolated.isna().any(axis=1).sum()
print(f"df_polynomial_interpolated Rows with at least one NaN: {num_na_rows}")

plot_VI_over_time_per_plot(plot_to_check, feature_name=features[0], vi_name=features[0].upper(), non_interpolated_dataset=df_hour13, with_interpolation=True, interpolation_method="Polynomial",
                           interpolated_dataset=df_polynomial_interpolated)
# Polynomial curve overshoot between points. Does not interpolate well for NDVI.

In [ ]:
# Apply spline interpolation
#without smoothing
df_spline_interpolated = (
    df_hour13
    .groupby('plot_id', group_keys=False)
    .apply(interpolate_plot, method='spline', features_to_interpolate=features)
    .reset_index(drop=True)
)
num_na_rows = df_spline_interpolated.isna().any(axis=1).sum()
print(f"df_spline_interpolated Rows with at least one NaN: {num_na_rows}")
# with smoothing
df_spline_interpolated_and_smoothed = (
    df_hour13
    .groupby('plot_id', group_keys=False)
    .apply(interpolate_plot, method='spline', features_to_interpolate=features, doSmoothing=True)
    .reset_index(drop=True)
)
num_na_rows = df_spline_interpolated_and_smoothed.isna().any(axis=1).sum()
print(f"df_spline_interpolated_and_smoothed Rows with at least one NaN: {num_na_rows}")

In [ ]:
#Piecewise Cubic Hermite (PCHIP) Interpolation
df_PCHIP_interpolated = (
    df_hour13
    .groupby('plot_id', group_keys=False)
    .apply(interpolate_plot, method='pchip', features_to_interpolate=features)
    .reset_index(drop=True)
)


In [ ]:
# Plot the NDVI over the time
plot_to_check = "2_d"
for feature in canopy_temp_features:
    plot_VI_over_time_per_plot(plot_to_check, feature, feature.upper(), df_hour13, with_interpolation=True, interpolation_method="Linear", interpolated_dataset=df_linear_interpolated)
    plot_VI_over_time_per_plot(plot_to_check, feature, feature.upper(), df_hour13, with_interpolation=True, interpolation_method="Linear Smoothed", interpolated_dataset=df_linear_interpolated_and_smoothed)
    plot_VI_over_time_per_plot(plot_to_check, feature, feature.upper(), df_hour13, with_interpolation=True, interpolation_method="Spline",
                               interpolated_dataset=df_spline_interpolated)
    plot_VI_over_time_per_plot(plot_to_check, feature, feature.upper(), df_hour13, with_interpolation=True, interpolation_method="Spline Smoothed",
                               interpolated_dataset=df_spline_interpolated_and_smoothed)
    plot_VI_over_time_per_plot(plot_to_check, feature, feature.upper(), df_hour13, with_interpolation=True, interpolation_method="PCHIP",
                               interpolated_dataset=df_PCHIP_interpolated)

In [ ]:
for feature in features:
    daily_corr = (
        df_linear_interpolated_and_smoothed.groupby('date')
        .apply(lambda g: g[feature].corr(g[output_variable]))
        .reset_index(name='correlation')
    )

    avg_corr = daily_corr['correlation'].mean()
    print(f"Average daily {feature.upper()}–Yield correlation: {avg_corr:.3f}")

# at 1pm
# Average daily NDVI_NOIR_RGB_METHOD–Yield correlation: 0.476
# Average daily GNDVI_NOIR_RGB_METHOD–Yield correlation: 0.466

# at 10am_mark
# Average daily NDVI_NOIR_RGB_METHOD–Yield correlation: 0.419
# Average daily GNDVI_NOIR_RGB_METHOD–Yield correlation: 0.445


# dfhour13
# Average daily NDVI_NOIR_RGB_METHOD–Yield correlation: 0.472
# Average daily GNDVI_NOIR_RGB_METHOD–Yield correlation: 0.460
# Average daily RDVI_NOIR_RGB_METHOD–Yield correlation: 0.442
# Average daily SAVI_NOIR_RGB_METHOD–Yield correlation: 0.427
# Average daily SR_NOIR_RGB_METHOD–Yield correlation: 0.479
# Average daily EVI_NOIR_RGB_METHOD–Yield correlation: 0.401
# Average daily CIGREEN_NOIR_RGB_METHOD–Yield correlation: 0.464
# Average daily GLI–Yield correlation: 0.188
# Average daily VARI–Yield correlation: 0.253
# Average daily LEPTON_MEAN_TEMP–Yield correlation: -0.053
# Average daily LEPTON_MIN_TEMP–Yield correlation: -0.049
# Average daily LEPTON_MAX_TEMP–Yield correlation: -0.031



In [ ]:
df_linear_interpolated_and_smoothed = df_linear_interpolated_and_smoothed[(df_linear_interpolated_and_smoothed['date'] >= '2024-05-01') & (df_linear_interpolated_and_smoothed['date'] <= '2024-06-15')]


In [ ]:
#Stratified split at plot level
#
# Get one yield per plot
plot_yields = df_linear_interpolated_and_smoothed.groupby('plot_id')[output_variable].first().reset_index()
# Create yield bins
plot_yields['yield_bin'] = pd.cut(plot_yields[output_variable], bins=5, labels=False)

from sklearn.model_selection import StratifiedShuffleSplit

train_plot_ids = []
test_plot_ids = []
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(plot_yields, plot_yields['yield_bin']):
    train_plot_ids = plot_yields.iloc[train_idx]['plot_id']
    test_plot_ids = plot_yields.iloc[test_idx]['plot_id']

In [ ]:
plot_yields

In [ ]:
train_df = df_linear_interpolated_and_smoothed[df_linear_interpolated_and_smoothed['plot_id'].isin(train_plot_ids)]
test_df  = df_linear_interpolated_and_smoothed[df_linear_interpolated_and_smoothed['plot_id'].isin(test_plot_ids)]

test_df

In [ ]:
print(set(train_df['plot_id']).intersection(set(test_df['plot_id'])))  # should be empty


In [ ]:

num_na_rows = train_df.isna().any(axis=1).sum()
print(f"train_df Rows with at least one NaN: {num_na_rows}")

num_na_rows = test_df.isna().any(axis=1).sum()
print(f"train_df Rows with at least one NaN: {num_na_rows}")
test_df

In [ ]:
with pd.ExcelWriter("data/stratified_train_test_datasets_v3_all_interpolated.xlsx", engine="openpyxl") as writer:
    train_df.to_excel(writer, sheet_name="Train", index=False)
    test_df.to_excel(writer, sheet_name="Test", index=False)
